# Notebook 04 — Model Training and Serialisation
Trains LR and XGBoost per city, saves 20 .joblib files, feature_columns.json, model_results.csv, and feature importance charts.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import json
import pandas as pd
import numpy as np
from src.features import FEATURE_COLS
from src.model import chronological_split, train_linear, train_xgboost, evaluate_model, save_model, load_and_verify
from src.visualisations import plot_feature_importance_city, plot_feature_importance_comparison, plot_model_comparison

FEATURED_CSV = '../data/processed/featured.csv'
MODELS_DIR   = '../models'
CHARTS_DIR   = '../charts'
RESULTS_CSV  = '../data/processed/model_results.csv'
FEAT_JSON    = '../models/feature_columns.json'

if not os.path.exists(FEATURED_CSV):
    raise FileNotFoundError(f'featured.csv not found. Run notebook 02 first.')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)

df = pd.read_csv(FEATURED_CSV, parse_dates=['date'])
TARGET_COL = 'aqi_target'
print(f'Loaded {len(df):,} rows. Feature cols: {FEATURE_COLS}')

In [ ]:
cities = sorted(df['city'].unique())
print(f'Cities ({len(cities)}): {cities}')

results = []
xgb_models = {}

for city in cities:
    city_df = df[df['city'] == city].sort_values('date').reset_index(drop=True)
    city_df = city_df.dropna(subset=FEATURE_COLS + [TARGET_COL])

    train_df, test_df = chronological_split(city_df, test_fraction=0.2)

    X_train = train_df[FEATURE_COLS].values
    y_train = train_df[TARGET_COL].values
    X_test  = test_df[FEATURE_COLS].values
    y_test  = test_df[TARGET_COL].values

    # Linear Regression
    lr = train_linear(X_train, y_train)
    lr_metrics = evaluate_model(lr, X_test, y_test)
    lr_path = os.path.join(MODELS_DIR, f'lr_{city.lower()}.joblib')
    save_model(lr, lr_path)
    results.append({'city': city, 'model_type': 'lr', **lr_metrics})

    # XGBoost
    try:
        xgb = train_xgboost(X_train, y_train)
        xgb_metrics = evaluate_model(xgb, X_test, y_test)
        xgb_path = os.path.join(MODELS_DIR, f'xgb_{city.lower()}.joblib')
        save_model(xgb, xgb_path)
        results.append({'city': city, 'model_type': 'xgb', **xgb_metrics})
        xgb_models[city] = xgb
    except Exception as e:
        print(f'WARNING: XGBoost failed for {city}: {e}')

    print(f'{city}: LR MAE={lr_metrics["mae"]:.2f} | XGB MAE={xgb_metrics["mae"]:.2f}')

print(f'\nTotal models saved: {len(results)}')

In [ ]:
# Verify round-trip fidelity for all saved models
all_ok = True
for city in cities:
    city_df = df[df['city'] == city].sort_values('date').reset_index(drop=True)
    city_df = city_df.dropna(subset=FEATURE_COLS + [TARGET_COL])
    _, test_df = chronological_split(city_df, test_fraction=0.2)
    X_test = test_df[FEATURE_COLS].values

    for model_type in ['lr', 'xgb']:
        path = os.path.join(MODELS_DIR, f'{model_type}_{city.lower()}.joblib')
        if not os.path.exists(path):
            print(f'MISSING: {path}')
            all_ok = False
            continue
        import joblib
        model = joblib.load(path)
        ok = load_and_verify(model, path, X_test)
        if not ok:
            print(f'MISMATCH: {path}')
            all_ok = False

print('Round-trip verification:', 'PASSED' if all_ok else 'FAILED')

In [ ]:
# Write feature_columns.json
with open(FEAT_JSON, 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f'Written {FEAT_JSON}: {FEATURE_COLS}')

# Write model_results.csv
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f'Written {RESULTS_CSV} ({len(results_df)} rows)')
print(results_df.to_string())

In [ ]:
# Generate feature importance charts (10 per-city + 1 comparison)
for city, model in xgb_models.items():
    p = plot_feature_importance_city(city, model, FEATURE_COLS, out_dir=CHARTS_DIR)
    print(f'Saved: {p}')

p = plot_feature_importance_comparison(xgb_models, FEATURE_COLS, out_dir=CHARTS_DIR)
print(f'Saved: {p}')

# Generate model comparison chart
p = plot_model_comparison(results_df, out_dir=CHARTS_DIR)
print(f'Saved: {p}')

In [ ]:
# Final artefact count
import glob
joblibs = glob.glob(os.path.join(MODELS_DIR, '*.joblib'))
pngs    = glob.glob(os.path.join(CHARTS_DIR, '*.png'))
print(f'.joblib files: {len(joblibs)} (expected 20)')
print(f'PNG files:     {len(pngs)} (expected 21: 9 EDA + 10 per-city + 1 comparison + 1 model_comparison)')
for j in sorted(joblibs): print(' ', os.path.basename(j))